## GRA4162 Deep Learning and Explainable AI
### Lab 10.1, Fall 2026: Autoencoders

#### Exercise 1: Building an autoencoder for feature extraction.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

# Load MNIST dataset
(x_train, _), (x_test, _) = tf.keras.datasets.mnist.load_data()

# Normalize and flatten images (0-1 range, shape: 28*28 = 784)
x_train = x_train.astype("float32") / 255.
x_test = x_test.astype("float32") / 255.
x_train = x_train.reshape((len(x_train), 28 * 28))
x_test = x_test.reshape((len(x_test), 28 * 28))

#observe the data shape:
print (x_train.shape)
print (x_test.shape)

In [ ]:
# Defining the size of the code (the bottleneck):
encoding_dim = 32  #modify to e.g. 64 or 128 to see the difference

# Defining the input layer:
input_img = tf.keras.Input(shape=(784,))

# Encoder layers:
encoded = layers.Dense(256, activation='relu')(input_img)
encoded = layers.Dense(128, activation='relu')(encoded)
encoded = layers.Dense(encoding_dim, activation='relu')(encoded)

# Decoder layers:
decoded = layers.Dense(128, activation='relu')(encoded)
decoded = layers.Dense(256, activation='sigmoid')(decoded)
decoded = layers.Dense(784, activation='sigmoid')(decoded)

# Autoencoder model
autoencoder = models.Model(input_img, decoded)

autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

# Train the model
autoencoder.fit(
    x_train, x_train,
    epochs=10, # try increasing the number
    batch_size=256,
    shuffle=True,
    validation_data=(x_test, x_test)
)

# Encode and decode some digits
encoded_imgs = autoencoder.predict(x_test)
decoded_imgs = encoded_imgs.reshape((-1, 28, 28))

# Visualize original vs reconstructed images
n = 10
plt.figure(figsize=(20, 4))
for i in range(n):
    # Original
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap="gray")
    plt.axis("off")

    # Reconstructed
    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(decoded_imgs[i], cmap="gray")
    plt.axis("off")

plt.show()

#### Exercise 2: Using the trained encoder in classification task.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
import matplotlib.pyplot as plt


# Load Fashion-MNIST

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
x_train = x_train.astype("float32") / 255.
x_test = x_test.astype("float32") / 255.

num_classes = 10
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


Build a model and summarise it.

In [ ]:

#Parameters

latent_dim = 64
input_shape = (28, 28)
num_classes = 10

#Build Encoder

encoder_inputs = tf.keras.Input(shape=input_shape)
x = layers.Flatten()(encoder_inputs)
latent = layers.Dense(latent_dim, activation='relu')(x)
encoder = Model(encoder_inputs, latent, name='encoder')


# Build Decoder

decoder_inputs = tf.keras.Input(shape=(latent_dim,))
x = layers.Dense(input_shape[0] * input_shape[1], activation='sigmoid')(decoder_inputs)
decoder_outputs = layers.Reshape(input_shape)(x)
decoder = Model(decoder_inputs, decoder_outputs, name='decoder')


# Build Autoencoder

autoencoder_inputs = tf.keras.Input(shape=input_shape)
encoded = encoder(autoencoder_inputs)
decoded = decoder(encoded)
autoencoder = Model(autoencoder_inputs, decoded, name='autoencoder')

# Compile it
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

encoder.summary()
decoder.summary()
autoencoder.summary()

In [ ]:

# Train Autoencoder

autoencoder.fit(
    x_train, x_train,
    epochs=20,
    batch_size=256,
    shuffle=True,
    validation_data=(x_test, x_test)
)


In [ ]:

# Freeze the encoder

encoder.trainable = False


# Build Classifier on top of Encoder

classifier_input = tf.keras.Input(shape=input_shape)
features = encoder(classifier_input)
x = layers.Dense(128, activation='relu')(features)
x = layers.Dense(num_classes, activation='softmax')(x)
classifier = Model(classifier_input, x, name='classifier')

classifier.compile(optimizer='adam',
                   loss='sparse_categorical_crossentropy',
                   metrics=['accuracy'])


# Train Classifier

classifier.fit(
    x_train, y_train,
    epochs=20,
    batch_size=256,
    validation_data=(x_test, y_test)
)


# Evaluate

test_loss, test_acc = classifier.evaluate(x_test, y_test)
print(f"Classifier accuracy: {test_acc:.4f}")


In [ ]:
import numpy as np

# Pick 10 test images

n = 10
x_subset = x_test[:n]
y_subset = y_test[:n]

# Autoencoder reconstruction
reconstructed = autoencoder.predict(x_subset)

# Classifier predictions
pred_probs = classifier.predict(x_subset)
pred_labels = np.argmax(pred_probs, axis=1)

# Plot results

plt.figure(figsize=(20, 6))
for i in range(n):
    # Original image
    ax = plt.subplot(3, n, i + 1)
    plt.imshow(x_subset[i], cmap='gray')
    plt.title("Orig")
    plt.axis('off')
    
    # Reconstructed image
    ax = plt.subplot(3, n, i + 1 + n)
    plt.imshow(reconstructed[i], cmap='gray')
    plt.title("Recon")
    plt.axis('off')
    
    # Predicted label
    ax = plt.subplot(3, n, i + 1 + 2*n)
    plt.imshow(x_subset[i], cmap='gray')
    plt.title(f"Pred: {class_names[pred_labels[i]]}")
    plt.axis('off')

plt.tight_layout()
plt.show()

#### Exercise 3: Denoising autoencoder.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, losses, Model
from tqdm.keras import TqdmCallback


# Define Encoder

encoder_inputs = tf.keras.Input(shape=(28, 28, 1))
x = layers.Conv2D(32, (3, 3), activation='relu', padding='same', strides=2)(encoder_inputs)
x = layers.Conv2D(8, (3, 3), activation='relu', padding='same', strides=2)(x)
encoder_outputs = x
encoder = Model(encoder_inputs, encoder_outputs, name='encoder')


# Define Decoder

decoder_inputs = tf.keras.Input(shape=(7, 7, 8))  # shape after encoder
x = layers.Conv2DTranspose(8, kernel_size=3, strides=2, activation='relu', padding='same')(decoder_inputs)
x = layers.Conv2DTranspose(32, kernel_size=3, strides=2, activation='relu', padding='same')(x)
decoder_outputs = layers.Conv2D(1, kernel_size=(3, 3), activation='sigmoid', padding='same')(x)
decoder = Model(decoder_inputs, decoder_outputs, name='decoder')


# Combine Encoder and Decoder

autoencoder_inputs = encoder_inputs
encoded = encoder(autoencoder_inputs)
decoded = decoder(encoded)
autoencoder = Model(autoencoder_inputs, decoded, name='autoencoder')

autoencoder.compile(optimizer='adam', loss=losses.MeanSquaredError())


# Load & Prepare Data (Fashion-MNIST)

(x_train, _), (x_test, _) = tf.keras.datasets.fashion_mnist.load_data()
x_train = x_train.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.
x_train = x_train[..., tf.newaxis]  # add channel dimension
x_test = x_test[..., tf.newaxis]

# Add noise
noise_factor = 0.4
x_train_noisy = x_train + noise_factor * tf.random.normal(shape=x_train.shape)
x_test_noisy = x_test + noise_factor * tf.random.normal(shape=x_test.shape)
x_train_noisy = tf.clip_by_value(x_train_noisy, 0., 1.) # limit the values to be in [0,1]
x_test_noisy = tf.clip_by_value(x_test_noisy, 0., 1.)


# Train the Autoencoder

autoencoder.fit(
    x_train_noisy, x_train,
    epochs=5,
    batch_size=128,
    shuffle=True,
    validation_data=(x_test_noisy, x_test),
    callbacks=[TqdmCallback(verbose=1)],
    verbose=0
)


# Inspect Encoder

encoder.summary()


# Visualize Results
#
import matplotlib.pyplot as plt
import numpy as np

n = 10
decoded_imgs = autoencoder.predict(x_test_noisy[:n])

plt.figure(figsize=(20, 4))
for i in range(n):
    # Noisy
    ax = plt.subplot(3, n, i + 1)
    plt.imshow(np.squeeze(x_test_noisy[i]), cmap="gray")
    plt.title("Noisy")
    plt.axis("off")

    # Denoised
    ax = plt.subplot(3, n, i + 1 + n)
    plt.imshow(np.squeeze(decoded_imgs[i]), cmap="gray")
    plt.title("Denoised")
    plt.axis("off")

    # Original
    ax = plt.subplot(3, n, i + 1 + 2*n)
    plt.imshow(np.squeeze(x_test[i]), cmap="gray")
    plt.title("Original")
    plt.axis("off")

plt.tight_layout()
plt.show()